# DSA-Mamba3 — Anemia Analysis (Kaggle)

**Two modes — switch with one line in the CONFIG cell:**
- `'task': 'classification'` → anemic / non-anemic  
- `'task': 'regression'`     → predict Hb value (g/dL)

**Run order:** Cell 1 (install) → Cell 2 (config) → Cell 3 (definitions) → Cell 4 (train + eval)

In [ ]:
# ── Cell 1: Install dependencies & clone repo ──────────────────────────────
import subprocess, sys, os

REPO_URL = 'https://github.com/junaidmaqbool/mambathree.git'
REPO_DIR = '/kaggle/working/mambathree'

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)
    print('Repo already present — pulled latest.')

# Pure-Python / Triton deps
for pkg in ['einops', 'timm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'triton'], check=True)
except Exception as e:
    print(f'triton warning: {e}')

try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'causal-conv1d'], check=True)
except Exception:
    print('causal-conv1d skipped — not needed for DSAmamba3.')

# Fix torchvision circular import (AttributeError: partially initialized module
# 'torchvision' has no attribute 'extension').  This is caused by a torch/torchvision
# version mismatch; upgrading torchvision to the latest compatible build resolves it.
import torch as _torch
_tv_map = {
    '2.0': '0.15.2',
    '2.1': '0.16.2',
    '2.2': '0.17.2',
    '2.3': '0.18.1',
    '2.4': '0.19.1',
    '2.5': '0.20.1',
    '2.6': '0.21.0',
}
_torch_maj_min = '.'.join(_torch.__version__.split('.')[:2])
_tv_pin = _tv_map.get(_torch_maj_min)
if _tv_pin:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    f'torchvision=={_tv_pin}'], check=True)
    print(f'torchvision pinned to {_tv_pin} for torch {_torch_maj_min}')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--upgrade', 'torchvision'], check=True)
    print('torchvision upgraded (torch version unrecognized, using latest)')

# ── Patch repo for Kaggle (no C-extension build) ─────────────────────────
# selective_scan_cuda is a compiled C++/CUDA extension only available after
# `pip install -e .` with nvcc.  On Kaggle we skip that build because
# DSAmamba3 uses Mamba3 (pure-Triton SISO kernels), not Mamba1.
# Wrap the bare imports in try/except so the package loads cleanly.

_init_path = os.path.join(REPO_DIR, 'mamba_ssm', '__init__.py')
_init_src = open(_init_path).read()
if 'selective_scan_fn = None' not in _init_src:
    print('Patching __init__.py  (add try/except for missing C extensions)…')
    with open(_init_path, 'w') as f:
        f.write('''\
__version__ = "2.3.1"

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn, mamba_inner_fn
    from mamba_ssm.modules.mamba_simple import Mamba
    from mamba_ssm.modules.mamba2 import Mamba2
    from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
except ImportError:
    selective_scan_fn = None
    mamba_inner_fn = None
    Mamba = None
    Mamba2 = None
    MambaLMHeadModel = None

try:
    from mamba_ssm.modules.mamba3 import Mamba3
except ImportError:
    Mamba3 = None
''')

_sscan_path = os.path.join(REPO_DIR, 'mamba_ssm', 'ops', 'selective_scan_interface.py')
_sscan_src = open(_sscan_path).read()
if 'selective_scan_cuda = None' not in _sscan_src:
    print('Patching selective_scan_interface.py  (guard selective_scan_cuda import)…')
    _sscan_src = _sscan_src.replace(
        'import selective_scan_cuda',
        'try:\n    import selective_scan_cuda\nexcept ImportError:\n    selective_scan_cuda = None',
        1,  # replace only the first bare occurrence
    )
    with open(_sscan_path, 'w') as f:
        f.write(_sscan_src)

# Add repo to sys.path (no pip install -e, no C++ build needed)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup complete.')


In [ ]:
# ── Cell 2: Configuration — edit EVERYTHING here ─────────────────────────

CONFIG = {
    # 'classification' → anemic vs non-anemic (CrossEntropyLoss)
    # 'regression'     → predict Hb value in g/dL (HuberLoss)
    'task': 'classification',

    # Dataset paths
    'dataset_dir': '/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye',
    'csv_path':    '/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.csv',

    # CSV column names
    'image_col':    'Patient ID',           # column with image filenames / IDs
    'hb_col':       'Haemoglobin (gm/dL)',  # numeric Hb values
    'label_col':    None,                   # None → auto-derive from hb_threshold
    'hb_threshold': 11.5,                   # g/dL (hb < threshold → anemic = 1)

    # Training
    'img_size':      224,
    'batch_size':    32,
    'num_epochs':    3,
    'learning_rate': 2e-4,
    'weight_decay':  0.05,
    'val_split':     0.2,
    'seed':          42,

    # Model — dims=[64,128,256,512] fits Kaggle T4; use [96,192,384,768] for full size
    'depths':         [2, 2, 4, 2],
    'dims':           [64, 128, 256, 512],
    'd_state':        32,    # reduce to 16 if OOM
    'headdim':        64,
    'expand':         2,
    'chunk_size':     64,
    'rope_fraction':  0.5,
    'drop_path_rate': 0.1,

    'checkpoint_dir': '/kaggle/working/checkpoints',
    'results_dir':    '/kaggle/working',
}

print('Task     :', CONFIG['task'])
print('img_size :', CONFIG['img_size'])
print('batch    :', CONFIG['batch_size'])
print('epochs   :', CONFIG['num_epochs'])


In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import os, sys, random, math, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

REPO_DIR = '/kaggle/working/mambathree'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Clear stale cache if re-running
for key in list(sys.modules.keys()):
    if key.startswith('mamba_ssm'):
        del sys.modules[key]

from mamba_ssm.models.DSAmamba3 import VSSM

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM   : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
# ── Cell 4: Dataset & DataLoaders ─────────────────────────────────────────

_IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}


def _build_image_index(image_dir: str) -> dict:
    """
    Scan image_dir recursively and return a dict:
        lowercase_stem  →  absolute_path

    This handles:
      • Any extension (jpg / jpeg / JPG / JPEG / png …)
      • Case differences between CSV IDs and actual filenames
      • Files in subdirectories
      • Extra suffixes — the CSV ID 'UIET-0001' will match
        'UIET-0001_L.jpg' because we strip the last _* suffix too.
    """
    index = {}
    for root, _, files in os.walk(image_dir):
        for fname in files:
            stem, ext = os.path.splitext(fname)
            if ext.lower() not in _IMG_EXTS:
                continue
            full = os.path.join(root, fname)
            # exact stem match (case-insensitive)
            index[stem.lower()] = full
            # also index with trailing _* suffix stripped
            # e.g. 'UIET-0003-0043241869_L' → 'uiet-0003-0043241869'
            if '_' in stem:
                stripped = stem.rsplit('_', 1)[0].lower()
                index.setdefault(stripped, full)
    return index


def _lookup(index: dict, filename: str) -> str:
    """Return resolved path from index, or None if not found."""
    key = str(filename).lower()
    if key in index:
        return index[key]
    # strip trailing _* the other way: CSV may have a suffix the files don't
    if '_' in key:
        stripped = key.rsplit('_', 1)[0]
        if stripped in index:
            return index[stripped]
    return None


def load_dataframe(cfg: dict) -> pd.DataFrame:
    """Load CSV, resolve image paths, drop any row that is incomplete."""
    df = pd.read_csv(cfg['csv_path'])

    missing_cols = [c for c in [cfg['image_col'], cfg['hb_col']] if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Columns not found: {missing_cols}.\n"
            f"Available columns: {list(df.columns)}\n"
            "Update image_col / hb_col in CONFIG to match your CSV."
        )

    df = df.rename(columns={cfg['image_col']: 'filename', cfg['hb_col']: 'hb'})

    # Derive binary label from Hb threshold if label column absent
    if cfg['label_col'] is None or cfg['label_col'] not in df.columns:
        thr = cfg['hb_threshold']
        df['hb'] = pd.to_numeric(df['hb'], errors='coerce')
        df['label'] = (df['hb'] < thr).astype('Int64')
        print(f'Label auto-derived: hb < {thr} g/dL → anemic (1)')
    else:
        df = df.rename(columns={cfg['label_col']: 'label'})
        df['hb'] = pd.to_numeric(df['hb'], errors='coerce')

    n0 = len(df)

    # Drop rows with NA filename, Hb, or label
    df = df.dropna(subset=['filename', 'hb', 'label'])
    na_dropped = n0 - len(df)

    # ── Build directory index once ───────────────────────────────────────
    image_dir = cfg['dataset_dir']
    print(f'Scanning image directory…  ({image_dir})')
    idx = _build_image_index(image_dir)
    print(f'  Found {len(idx)} unique image stems on disk.')

    # Resolve each row to an absolute path; None means file missing
    df = df.copy()
    df['img_path'] = df['filename'].apply(lambda fn: _lookup(idx, fn))
    img_dropped = int(df['img_path'].isna().sum())
    df = df[df['img_path'].notna()].reset_index(drop=True)

    # Report
    if na_dropped:
        print(f'Dropped {na_dropped:,} rows — NA filename / Hb / label')
    if img_dropped:
        print(f'Dropped {img_dropped:,} rows — image not found on disk')

    if len(df) == 0:
        sample_files = os.listdir(image_dir)[:10] if os.path.isdir(image_dir) else []
        raise RuntimeError(
            "No valid samples remain after filtering!\n"
            f"  dataset_dir  : {image_dir}\n"
            f"  First 10 files: {sample_files}\n"
            "Check dataset_dir and image_col in CONFIG."
        )

    print(f'Valid samples : {len(df):,}')
    if cfg['task'] == 'classification':
        counts = df['label'].value_counts().sort_index()
        print(f'  non-anemic (0): {counts.get(0, 0):,}')
        print(f'  anemic     (1): {counts.get(1, 0):,}')
    else:
        print(f'  Hb range : {df["hb"].min():.1f} – {df["hb"].max():.1f} g/dL'
              f'  (mean {df["hb"].mean():.1f})')
    return df


class AnemiaDataset(Dataset):
    """
    Dataset for anemia classification / Hb regression.

    img_path is pre-resolved at load time so __getitem__ never does I/O
    path searching.  Corrupted images are skipped and the next valid
    sample (by index) is returned instead.
    """

    def __init__(self, df: pd.DataFrame, transform, task: str = 'classification'):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.task      = task

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        for offset in range(len(self.df)):
            real_idx = (idx + offset) % len(self.df)
            row = self.df.iloc[real_idx]
            try:
                img = Image.open(row['img_path']).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                if self.task == 'classification':
                    return img, int(row['label'])
                else:
                    return img, float(row['hb'])
            except Exception:
                continue
        raise RuntimeError(
            f'No loadable image found in dataset (tried all {len(self.df)} rows)'
        )


def build_transforms(img_size: int):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, val_tf


# ── Build splits ──────────────────────────────────────────────────────────
df = load_dataframe(CONFIG)

stratify = df['label'] if CONFIG['task'] == 'classification' else None
train_df, val_df = train_test_split(
    df,
    test_size=CONFIG['val_split'],
    random_state=CONFIG['seed'],
    stratify=stratify,
)
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}')

train_tf, val_tf = build_transforms(CONFIG['img_size'])

# AnemiaDataset now receives the df with pre-resolved img_path — no image_dir needed
train_ds = AnemiaDataset(train_df, train_tf, CONFIG['task'])
val_ds   = AnemiaDataset(val_df,   val_tf,   CONFIG['task'])

train_loader = DataLoader(
    train_ds, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=2, pin_memory=True,
)
print(f'Batches → train: {len(train_loader)}  val: {len(val_loader)}')

In [ ]:
# ── Cell 5: Model sanity check ────────────────────────────────────────────
# num_classes=2 for classification → (B,2) logits → CrossEntropyLoss
# num_classes=1 for regression     → (B,1) scalar → HuberLoss (squeezed to (B,))

num_classes = 2 if CONFIG['task'] == 'classification' else 1

model = VSSM(
    patch_size     = 4,
    in_chans       = 3,
    num_classes    = num_classes,
    depths         = CONFIG['depths'],
    dims           = CONFIG['dims'],
    d_state        = CONFIG['d_state'],
    headdim        = CONFIG['headdim'],
    expand         = CONFIG['expand'],
    chunk_size     = CONFIG['chunk_size'],
    rope_fraction  = CONFIG['rope_fraction'],
    drop_path_rate = CONFIG['drop_path_rate'],
).to(DEVICE)

print(f'Parameters : {model.num_parameters():,}')
print(f'Task       : {CONFIG["task"]}   num_classes={num_classes}')

# Sanity-check forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, CONFIG['img_size'], CONFIG['img_size']).to(DEVICE)
    out   = model(dummy)
    print(f'Forward OK  input {tuple(dummy.shape)} → output {tuple(out.shape)}')

# Free the sanity-check model so Cell 7 can allocate a fresh one without OOM
del dummy, out, model
torch.cuda.empty_cache()

In [ ]:
# ── Cell 6: Training utilities ────────────────────────────────────────────

def make_optimizer(model, lr: float, wd: float):
    """AdamW with weight-decay skipped for Mamba3 special parameters."""
    no_decay = model.no_weight_decay()   # {'dt_bias', 'D', 'B_bias', 'C_bias'}
    decay_params, nodecay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(name.endswith(nd) for nd in no_decay):
            nodecay_params.append(p)
        else:
            decay_params.append(p)
    return torch.optim.AdamW(
        [{'params': decay_params,   'weight_decay': wd},
         {'params': nodecay_params, 'weight_decay': 0.0}],
        lr=lr,
    )


def train_one_epoch(model, loader, optimizer, loss_fn, device, task):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        if task == 'regression':
            loss = loss_fn(outputs.squeeze(1), targets.float())
        else:
            loss = loss_fn(outputs, targets.long())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, loss_fn, device, task):
    model.eval()
    total_loss   = 0.0
    all_preds    = []
    all_targets  = []
    for images, targets in loader:
        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        outputs = model(images)
        if task == 'regression':
            loss = loss_fn(outputs.squeeze(1), targets.float())
            all_preds.extend(outputs.squeeze(1).cpu().numpy())
        else:
            loss = loss_fn(outputs, targets.long())
            all_preds.extend(outputs.argmax(1).cpu().numpy())
        total_loss += loss.item() * images.size(0)
        all_targets.extend(targets.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_preds), np.array(all_targets)


def compute_metrics(preds, targets, task):
    if task == 'classification':
        acc = accuracy_score(targets.astype(int), preds.astype(int)) * 100
        f1  = f1_score(targets.astype(int), preds.astype(int), average='binary') * 100
        return {'accuracy': acc, 'f1': f1}
    else:
        mae  = float(np.mean(np.abs(preds - targets)))
        rmse = float(np.sqrt(np.mean((preds - targets) ** 2)))
        ss_res = float(np.sum((targets - preds) ** 2))
        ss_tot = float(np.sum((targets - targets.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
        return {'mae': mae, 'rmse': rmse, 'r2': r2}


print('Training utilities ready.')

In [ ]:
# ── Cell 7: Training loop ─────────────────────────────────────────────────

# Build model here so this cell is always self-contained
num_classes = 2 if CONFIG['task'] == 'classification' else 1
model = VSSM(
    patch_size=4, in_chans=3, num_classes=num_classes,
    depths=CONFIG['depths'], dims=CONFIG['dims'],
    d_state=CONFIG['d_state'], headdim=CONFIG['headdim'],
    expand=CONFIG['expand'], chunk_size=CONFIG['chunk_size'],
    rope_fraction=CONFIG['rope_fraction'],
    drop_path_rate=CONFIG['drop_path_rate'],
).to(DEVICE)
print(f'Parameters : {model.num_parameters():,}')

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

optimizer = make_optimizer(model, CONFIG['learning_rate'], CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['num_epochs'], eta_min=1e-6
)

if CONFIG['task'] == 'classification':
    loss_fn        = nn.CrossEntropyLoss()
    monitor_metric = 'f1'
    higher_better  = True
else:
    loss_fn        = nn.HuberLoss(delta=1.0)
    monitor_metric = 'mae'
    higher_better  = False

history    = {'train_loss': [], 'val_loss': [], 'val_metric': []}
best_val   = -float('inf') if higher_better else float('inf')
best_epoch = 0

hdr = f"{'Epoch':>5}  {'TrainLoss':>9}  {'ValLoss':>8}  {monitor_metric.upper():>10}  {'LR':>8}  Time"
sep = '─' * len(hdr)
print(hdr); print(sep)

for epoch in range(1, CONFIG['num_epochs'] + 1):
    t0 = time.time()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, loss_fn, DEVICE, CONFIG['task']
    )
    val_loss, preds, targets_np = evaluate(
        model, val_loader, loss_fn, DEVICE, CONFIG['task']
    )
    metrics = compute_metrics(preds, targets_np, CONFIG['task'])
    scheduler.step()

    val_metric = metrics[monitor_metric]
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_metric'].append(val_metric)

    improved = (higher_better and val_metric > best_val) or \
               (not higher_better and val_metric < best_val)
    if improved:
        best_val   = val_metric
        best_epoch = epoch
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'config':          CONFIG,
            'best_val':        best_val,
            'monitor_metric':  monitor_metric,
        }, os.path.join(CONFIG['checkpoint_dir'], 'best.pth'))
        star = ' *'
    else:
        star = ''

    lr_now  = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0
    unit    = '%' if CONFIG['task'] == 'classification' else ''
    print(f"{epoch:>5}  {train_loss:>9.4f}  {val_loss:>8.4f}  "
          f"{val_metric:>9.3f}{unit}  {lr_now:>8.2e}  {elapsed:.0f}s{star}")

print(sep)
print(f'Best epoch: {best_epoch}  |  Best val {monitor_metric}: {best_val:.4f}')


In [ ]:
# ── Cell 8: Evaluate best checkpoint ─────────────────────────────────────
ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'best.pth')
ckpt      = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best checkpoint  epoch={ckpt['epoch']}  "
      f"{ckpt['monitor_metric']}={ckpt['best_val']:.4f}")

_, preds, targets_np = evaluate(model, val_loader, loss_fn, DEVICE, CONFIG['task'])
metrics = compute_metrics(preds, targets_np, CONFIG['task'])

print('\n── Validation metrics ──────────────────────')
for k, v in metrics.items():
    unit = '%' if k in ('accuracy', 'f1') else ''
    print(f'  {k:<12}: {v:.4f}{unit}')

if CONFIG['task'] == 'classification':
    print('\n── Classification report ───────────────────')
    print(classification_report(
        targets_np.astype(int), preds.astype(int),
        target_names=['non-anemic', 'anemic'],
    ))

In [ ]:
# ── Cell 9: Visualise results ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"DSA-Mamba3 — {CONFIG['task'].capitalize()}", fontsize=14)

# ── Loss curves ──────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(history['train_loss'], label='train')
ax.plot(history['val_loss'],   label='val')
ax.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.6, label=f'best={best_epoch}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Loss curves'); ax.legend()

# ── Metric curve ─────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(history['val_metric'])
ax.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.6)
unit = ' %' if CONFIG['task'] == 'classification' else ' g/dL'
ax.set_xlabel('Epoch')
ax.set_ylabel(f'{monitor_metric}{unit}')
ax.set_title(f'Val {monitor_metric}')

# ── Task-specific panel ───────────────────────────────────────────────────
ax = axes[2]
if CONFIG['task'] == 'classification':
    cm = confusion_matrix(targets_np.astype(int), preds.astype(int))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=['non-anemic', 'anemic'],
                yticklabels=['non-anemic', 'anemic'],
                cmap='Blues')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion matrix  (acc={metrics["accuracy"]:.1f}%)')
else:
    ax.scatter(targets_np, preds, alpha=0.35, s=10, color='steelblue')
    lo = min(float(targets_np.min()), float(preds.min())) - 0.5
    hi = max(float(targets_np.max()), float(preds.max())) + 0.5
    ax.plot([lo, hi], [lo, hi], 'r--', label='ideal')
    ax.set_xlabel('Actual Hb (g/dL)'); ax.set_ylabel('Predicted Hb (g/dL)')
    ax.set_title(f'Scatter  MAE={metrics["mae"]:.2f}  R²={metrics["r2"]:.3f}')
    ax.legend()

plt.tight_layout()
out_png = os.path.join(CONFIG['results_dir'], f'results_{CONFIG["task"]}.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}')

## Quick reference

| Goal | Change in CONFIG |
|------|------------------|
| Switch to regression | `'task': 'regression'` then re-run cells 4–9 |
| Larger model (needs ~14 GB) | `'dims': [96, 192, 384, 768], 'd_state': 64` |
| OOM / out-of-memory | Halve `batch_size`, or use `'dims': [48, 96, 192, 384]` |
| Grayscale images | `in_chans=1` in Cell 5 model block |
| Custom Hb threshold | `'hb_threshold': 11.0` (WHO female threshold) |
| Skip label column | `'label_col': None` — auto-derived from `hb_threshold` |

### Output files
- `/kaggle/working/checkpoints/best.pth` — best model checkpoint
- `/kaggle/working/results_classification.png` / `results_regression.png`

### Repo structure reminder
```
mambathree/
  mamba_ssm/models/DSAmamba3.py   ← VSSM model (our backbone)
  mamba_ssm/modules/mamba3.py     ← official Mamba3 SSM core
```

---
# Part 2 — Mamba3Vision

A **simpler, flat** alternative backbone — single `dim` throughout, no VMamba hierarchy.  
Reuses the same dataset loaders and training utilities defined above.

| | DSA-Mamba3 (Part 1) | Mamba3Vision (Part 2) |
|---|---|---|
| Architecture | Hierarchical 4 stages | Flat N identical blocks |
| Spatial tokens | Shrinks per stage | Fixed (196 for patch=16) |
| Params | ~25 M | ~7 M |

Run **Cell 10 → Cell 16** after finishing Part 1.

In [ ]:
# ── Cell 10: Mamba3Vision config ──────────────────────────────────────────
# Inherits dataset / training settings from CONFIG above.
# Change only the model knobs here; task/dataset/epochs are shared.

M3V_CONFIG = {
    **CONFIG,   # inherit everything from DSA-Mamba3 config

    # Override model-specific fields for Mamba3Vision
    'patch_size':      16,   # 224/16 = 14×14 = 196 tokens
    'dim':             256,  # single channel width throughout
    'depth':           6,    # number of Mamba3 blocks
    # d_state, headdim, expand, chunk_size, rope_fraction, drop_path_rate
    # are kept from CONFIG

    'checkpoint_dir': '/kaggle/working/checkpoints_m3v',
    'results_dir':    '/kaggle/working',
}

# Import Mamba3Vision (model file lives in the cloned repo)
import sys, os
REPO_DIR = '/kaggle/working/mambathree'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Clear stale module cache in case Part 1 already imported mamba_ssm
for _k in list(sys.modules.keys()):
    if _k.startswith('mamba_ssm'):
        del sys.modules[_k]

from mamba_ssm.models.mamba3_vision import Mamba3Vision

print('Task       :', M3V_CONFIG['task'])
print('patch_size :', M3V_CONFIG['patch_size'],
      f"→ {(M3V_CONFIG['img_size'] // M3V_CONFIG['patch_size'])**2} tokens")
print('dim        :', M3V_CONFIG['dim'], '  depth:', M3V_CONFIG['depth'])

In [ ]:
# ── Cell 11: Mamba3Vision sanity check ────────────────────────────────────
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_num_classes_m3v = 2 if M3V_CONFIG['task'] == 'classification' else 1

_m3v_sanity = Mamba3Vision(
    img_size       = M3V_CONFIG['img_size'],
    patch_size     = M3V_CONFIG['patch_size'],
    in_chans       = 3,
    num_classes    = _num_classes_m3v,
    dim            = M3V_CONFIG['dim'],
    depth          = M3V_CONFIG['depth'],
    d_state        = M3V_CONFIG['d_state'],
    headdim        = M3V_CONFIG['headdim'],
    expand         = M3V_CONFIG['expand'],
    chunk_size     = M3V_CONFIG['chunk_size'],
    rope_fraction  = M3V_CONFIG['rope_fraction'],
    drop_path_rate = M3V_CONFIG['drop_path_rate'],
).to(DEVICE)

print(f'Parameters : {_m3v_sanity.num_parameters():,}')

with torch.no_grad():
    _dummy = torch.randn(2, 3, M3V_CONFIG['img_size'], M3V_CONFIG['img_size']).to(DEVICE)
    _out   = _m3v_sanity(_dummy)
    print(f'Forward OK  {tuple(_dummy.shape)} → {tuple(_out.shape)}')

del _dummy, _out, _m3v_sanity
torch.cuda.empty_cache()

In [ ]:
# ── Cell 12: Train Mamba3Vision ────────────────────────────────────────────
import os, time
import torch
import torch.nn as nn

num_classes_m3v = 2 if M3V_CONFIG['task'] == 'classification' else 1

model_m3v = Mamba3Vision(
    img_size=M3V_CONFIG['img_size'],   patch_size=M3V_CONFIG['patch_size'],
    in_chans=3,                        num_classes=num_classes_m3v,
    dim=M3V_CONFIG['dim'],             depth=M3V_CONFIG['depth'],
    d_state=M3V_CONFIG['d_state'],     headdim=M3V_CONFIG['headdim'],
    expand=M3V_CONFIG['expand'],       chunk_size=M3V_CONFIG['chunk_size'],
    rope_fraction=M3V_CONFIG['rope_fraction'],
    drop_path_rate=M3V_CONFIG['drop_path_rate'],
).to(DEVICE)
print(f'Parameters : {model_m3v.num_parameters():,}')

os.makedirs(M3V_CONFIG['checkpoint_dir'], exist_ok=True)

# AdamW — skip weight decay on pos_embed and Mamba3 special params
_no_decay_m3v = model_m3v.no_weight_decay()
_decay_p, _nodecay_p = [], []
for _n, _p in model_m3v.named_parameters():
    if not _p.requires_grad:
        continue
    if any(_n.endswith(_nd) for _nd in _no_decay_m3v) or 'pos_embed' in _n:
        _nodecay_p.append(_p)
    else:
        _decay_p.append(_p)
optimizer_m3v = torch.optim.AdamW(
    [{'params': _decay_p,   'weight_decay': M3V_CONFIG['weight_decay']},
     {'params': _nodecay_p, 'weight_decay': 0.0}],
    lr=M3V_CONFIG['learning_rate'],
)
scheduler_m3v = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_m3v, T_max=M3V_CONFIG['num_epochs'], eta_min=1e-6
)

if M3V_CONFIG['task'] == 'classification':
    loss_fn_m3v        = nn.CrossEntropyLoss()
    monitor_metric_m3v = 'f1'
    higher_better_m3v  = True
else:
    loss_fn_m3v        = nn.HuberLoss(delta=1.0)
    monitor_metric_m3v = 'mae'
    higher_better_m3v  = False

history_m3v    = {'train_loss': [], 'val_loss': [], 'val_metric': []}
best_val_m3v   = -float('inf') if higher_better_m3v else float('inf')
best_epoch_m3v = 0

hdr = (f"{'Epoch':>5}  {'TrainLoss':>9}  {'ValLoss':>8}  "
       f"{monitor_metric_m3v.upper():>10}  {'LR':>8}  Time")
sep = '─' * len(hdr)
print(hdr); print(sep)

for epoch in range(1, M3V_CONFIG['num_epochs'] + 1):
    t0 = time.time()
    train_loss = train_one_epoch(
        model_m3v, train_loader, optimizer_m3v, loss_fn_m3v, DEVICE, M3V_CONFIG['task']
    )
    val_loss, preds_m3v, targets_m3v = evaluate(
        model_m3v, val_loader, loss_fn_m3v, DEVICE, M3V_CONFIG['task']
    )
    metrics_m3v = compute_metrics(preds_m3v, targets_m3v, M3V_CONFIG['task'])
    scheduler_m3v.step()

    val_metric = metrics_m3v[monitor_metric_m3v]
    history_m3v['train_loss'].append(train_loss)
    history_m3v['val_loss'].append(val_loss)
    history_m3v['val_metric'].append(val_metric)

    improved = (higher_better_m3v and val_metric > best_val_m3v) or \
               (not higher_better_m3v and val_metric < best_val_m3v)
    if improved:
        best_val_m3v   = val_metric
        best_epoch_m3v = epoch
        torch.save({
            'epoch': epoch, 'model_state': model_m3v.state_dict(),
            'optimizer_state': optimizer_m3v.state_dict(),
            'config': M3V_CONFIG, 'best_val': best_val_m3v,
            'monitor_metric': monitor_metric_m3v,
        }, os.path.join(M3V_CONFIG['checkpoint_dir'], 'best.pth'))
        star = ' *'
    else:
        star = ''

    lr_now  = scheduler_m3v.get_last_lr()[0]
    elapsed = time.time() - t0
    unit    = '%' if M3V_CONFIG['task'] == 'classification' else ''
    print(f"{epoch:>5}  {train_loss:>9.4f}  {val_loss:>8.4f}  "
          f"{val_metric:>9.3f}{unit}  {lr_now:>8.2e}  {elapsed:.0f}s{star}")

print(sep)
print(f'Best epoch: {best_epoch_m3v}  |  Best val {monitor_metric_m3v}: {best_val_m3v:.4f}')

In [ ]:
# ── Cell 13: Evaluate Mamba3Vision best checkpoint ────────────────────────
import torch

ckpt_m3v  = torch.load(
    os.path.join(M3V_CONFIG['checkpoint_dir'], 'best.pth'),
    map_location=DEVICE, weights_only=False,
)
model_m3v.load_state_dict(ckpt_m3v['model_state'])
print(f"Loaded best checkpoint  epoch={ckpt_m3v['epoch']}  "
      f"{ckpt_m3v['monitor_metric']}={ckpt_m3v['best_val']:.4f}")

_, preds_m3v, targets_m3v = evaluate(
    model_m3v, val_loader, loss_fn_m3v, DEVICE, M3V_CONFIG['task']
)
metrics_m3v = compute_metrics(preds_m3v, targets_m3v, M3V_CONFIG['task'])

print('\n── Mamba3Vision — Validation metrics ──────────')
for k, v in metrics_m3v.items():
    unit = '%' if k in ('accuracy', 'f1') else ''
    print(f'  {k:<12}: {v:.4f}{unit}')

if M3V_CONFIG['task'] == 'classification':
    print('\n── Classification report ───────────────────')
    from sklearn.metrics import classification_report
    print(classification_report(
        targets_m3v.astype(int), preds_m3v.astype(int),
        target_names=['non-anemic', 'anemic'],
    ))

In [ ]:
# ── Cell 14: Visualise Mamba3Vision results ────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Mamba3Vision — {M3V_CONFIG['task'].capitalize()}", fontsize=14)

ax = axes[0]
ax.plot(history_m3v['train_loss'], label='train')
ax.plot(history_m3v['val_loss'],   label='val')
ax.axvline(best_epoch_m3v - 1, color='red', linestyle='--', alpha=0.6,
           label=f'best={best_epoch_m3v}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Loss curves'); ax.legend()

ax = axes[1]
ax.plot(history_m3v['val_metric'])
ax.axvline(best_epoch_m3v - 1, color='red', linestyle='--', alpha=0.6)
unit = ' %' if M3V_CONFIG['task'] == 'classification' else ' g/dL'
ax.set_xlabel('Epoch')
ax.set_ylabel(f'{monitor_metric_m3v}{unit}')
ax.set_title(f'Val {monitor_metric_m3v}')

ax = axes[2]
if M3V_CONFIG['task'] == 'classification':
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(targets_m3v.astype(int), preds_m3v.astype(int))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=['non-anemic', 'anemic'],
                yticklabels=['non-anemic', 'anemic'],
                cmap='Greens')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion matrix  (acc={metrics_m3v["accuracy"]:.1f}%)')
else:
    ax.scatter(targets_m3v, preds_m3v, alpha=0.35, s=10, color='seagreen')
    lo = min(float(targets_m3v.min()), float(preds_m3v.min())) - 0.5
    hi = max(float(targets_m3v.max()), float(preds_m3v.max())) + 0.5
    ax.plot([lo, hi], [lo, hi], 'r--', label='ideal')
    ax.set_xlabel('Actual Hb (g/dL)'); ax.set_ylabel('Predicted Hb (g/dL)')
    ax.set_title(f'Scatter  MAE={metrics_m3v["mae"]:.2f}  R²={metrics_m3v["r2"]:.3f}')
    ax.legend()

plt.tight_layout()
out_png = os.path.join(M3V_CONFIG['results_dir'], f'results_m3v_{M3V_CONFIG["task"]}.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}')

In [ ]:
# ── Cell 15: Side-by-side comparison ─────────────────────────────────────
import pandas as pd

task = CONFIG['task']      # same for both models

if task == 'classification':
    _metric_key = 'f1'
    _unit = '%'
else:
    _metric_key = 'mae'
    _unit = ' g/dL'

# Collect final val metrics for both models
# (preds/targets_np from Part 1 Cell 8; preds_m3v/targets_m3v from Part 2 Cell 13)
try:
    row_dsa = {k: f"{v:.4f}{_unit if k in ('f1','accuracy','mae','rmse') else ''}"
               for k, v in metrics.items()}
    row_m3v = {k: f"{v:.4f}{_unit if k in ('f1','accuracy','mae','rmse') else ''}"
               for k, v in metrics_m3v.items()}
    comp = pd.DataFrame([row_dsa, row_m3v],
                        index=['DSA-Mamba3 (hierarchical)', 'Mamba3Vision (flat)'])
    print('══ Comparison ══════════════════════════════════')
    print(comp.to_string())
    print()
except NameError:
    print("Run Part 1 Cell 8 first to get DSA-Mamba3 metrics (`metrics` variable).")

# Training curve overlay
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'DSA-Mamba3 vs Mamba3Vision — {task.capitalize()}', fontsize=13)

ax = axes[0]
ax.plot(history['val_loss'],     label='DSA-Mamba3', color='steelblue')
ax.plot(history_m3v['val_loss'], label='Mamba3Vision', color='seagreen')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val loss')
ax.set_title('Validation loss'); ax.legend()

ax = axes[1]
ax.plot(history['val_metric'],     label='DSA-Mamba3', color='steelblue')
ax.plot(history_m3v['val_metric'], label='Mamba3Vision', color='seagreen')
ax.set_xlabel('Epoch')
ax.set_ylabel(f'{_metric_key.upper()}{_unit}')
ax.set_title(f'Val {_metric_key.upper()}'); ax.legend()

plt.tight_layout()
out_cmp = os.path.join(CONFIG['results_dir'], f'comparison_{task}.png')
plt.savefig(out_cmp, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_cmp}')